In [7]:
%%capture
!pip install setuptools==68.0.0 ydata-profiling scikit-learn pandas numpy missingno


In [8]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [9]:
from ydata_profiling import ProfileReport
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns",25)
pd.set_option("display.max_rows",60)
sns.set_style("whitegrid")
print("Libraries imported successfully!")

Libraries imported successfully!


#########################################################################################################
CREACION DE MODELOS
#########################################################################################################



In [12]:
# Cargar datos limpios
ruta = "datos_limpios.csv"
df = pd.read_csv(ruta)

# Variables predictoras y objetivo
features = ["Flight phase", "Flight type", "Crash site", "Crew on board", "Pax on board", "Crash cause"]
target = "Total fatalities"

X = df[features]
y = df[target]

# División de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Preprocesamiento: escalado numérico + codificación de categóricas
numeric_features = ["Crew on board", "Pax on board"]
cat_features = ["Flight phase", "Flight type", "Crash site", "Crash cause"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_features,
        ),
    ],
    remainder="drop",
)

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []

#Print de un cuadro con el f-measure de cada modelo
for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    f1_score = report["weighted avg"]["f1-score"]
    results.append((name, acc, f1_score))
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1 Score"])
print(results_df)

            Model  Accuracy  F1 Score
0   Decision Tree  0.821429  0.819402
1             KNN  0.739286  0.737096
2  Neural Network  0.782143  0.781687
3             SVM  0.771429  0.768089
4   Random Forest  0.839286  0.838338


In [15]:
results = []

for name, model in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results.append({"Model": name, "Accuracy": accuracy, "Report": report})
    print(f"Modelo: {name}")
    print(f"  - Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("-" * 80)
    
summary = pd.DataFrame([{"Model": r["Model"], "Accuracy": r["Accuracy"]} for r in results])
print("\nResumen de exactitud de los modelos:")
print(summary.sort_values(by="Accuracy", ascending=False).reset_index(drop=True))

Modelo: Decision Tree
  - Accuracy: 0.8214
              precision    recall  f1-score   support

        0-10       0.83      0.71      0.76        95
        101+       0.93      1.00      0.96        40
       11-40       0.77      0.84      0.81       103
      41-100       0.84      0.86      0.85        42

    accuracy                           0.82       280
   macro avg       0.84      0.85      0.84       280
weighted avg       0.82      0.82      0.82       280

--------------------------------------------------------------------------------
Modelo: KNN
  - Accuracy: 0.7393
              precision    recall  f1-score   support

        0-10       0.73      0.65      0.69        95
        101+       0.81      0.97      0.89        40
       11-40       0.68      0.75      0.71       103
      41-100       0.85      0.69      0.76        42

    accuracy                           0.74       280
   macro avg       0.77      0.77      0.76       280
weighted avg       0.74     

Random Forest:  Este modelo es el que ofrece el mejor rendimiento con una exactitud del 83.93%.
                Como ya he mencionado en otros trabajos, este es el mejor modelo ya que la gran cantidad
                de analisis variable a variable, determinando todas las condiciones especificas de
                cada caso, permitiendo así clasificarlos de manera mas efectiva.

Decision Tree:  Con una exactitud del 82.14%, mantiene misma idea del anterior, solo que al ser solo un arbol
                tiene menos reglas a aplicar

Neural Network (Red Neuronal): Alcanza una exactitud del 78.21%. Aunque es un resultado razonable, está por debajo
                de los modelos como Random Forest y Decision Tree.

SVM (Support Vector Machine): Ofrece una exactitud del 77.14%.

KNN (K-Nearest Neighbors): Es el modelo con el rendimiento más bajo, registrando una exactitud del 73.93%.

In [16]:
# Hiperparametrización de Random Forest con GridSearch
from sklearn.model_selection import GridSearchCV

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])

param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [None, 5, 10, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Mejores parámetros encontrados para Random Forest:")
print(grid_search.best_params_)

best_rf = grid_search.best_estimator_

y_pred_rf = best_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy Random Forest optimizado: {accuracy_rf:.4f}")
print(classification_report(y_test, y_pred_rf, zero_division=0))


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Mejores parámetros encontrados para Random Forest:
{'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}
Accuracy Random Forest optimizado: 0.8393
              precision    recall  f1-score   support

        0-10       0.91      0.73      0.81        95
        101+       0.85      1.00      0.92        40
       11-40       0.80      0.87      0.83       103
      41-100       0.82      0.86      0.84        42

    accuracy                           0.84       280
   macro avg       0.84      0.86      0.85       280
weighted avg       0.85      0.84      0.84       280



Modelo: Random Forest SIN OPTIMIZAR
  - Accuracy: 0.8393
              precision    recall  f1-score   support

        0-10       0.83      0.82      0.83        95
        101+       0.91      1.00      0.95        40
       11-40       0.83      0.81      0.82       103
      41-100       0.81      0.81      0.81        42

    accuracy                           0.84       280
   macro avg       0.84      0.86      0.85       280
weighted avg       0.84      0.84      0.84       280

In [17]:
import joblib

# Guardar el modelo optimizado de Random Forest
modelo_path = "random_forest_model.pkl"
joblib.dump(best_rf, modelo_path)

print(f"Modelo Random Forest guardado exitosamente en: {modelo_path}")

# Verificar que se guardó correctamente cargándolo
modelo_cargado = joblib.load(modelo_path)
print(f"Modelo cargado correctamente: {type(modelo_cargado)}")

# Hacer una predicción de prueba con el modelo cargado
y_pred_prueba = modelo_cargado.predict(X_test[:5])
print(f"\nPredicción de prueba con modelo cargado: {y_pred_prueba}")

Modelo Random Forest guardado exitosamente en: random_forest_model.pkl
Modelo cargado correctamente: <class 'sklearn.pipeline.Pipeline'>

Predicción de prueba con modelo cargado: ['11-40' '0-10' '0-10' '11-40' '0-10']
